# 체인 직렬화(Serialization) 실습

LangChain의 프롬프트·모델·체인 객체를 **파일로 저장했다가 나중에 다시 불러와서 그대로 사용**하는 방법을 실습한다. 객체를 그대로 저장 가능한 형태(dict, JSON 문자열)로 바꾸는 것을 "직렬화(serialize)", 저장된 것을 다시 원래 객체로 복원하는 것을 "역직렬화(deserialize)"라고 부른다.

## 1. 환경변수 로드

`.env`에 저장된 `OPENAI_API_KEY`를 불러온다.

In [18]:
from dotenv import load_dotenv

# .env 파일의 OPENAI_API_KEY 등 환경변수를 불러온다.
load_dotenv()

True

## 2. 프롬프트 준비

"{fruit}의 색상이 무엇입니까?"라는 간단한 프롬프트 템플릿을 만든다.

In [19]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("{fruit}의 색상이 무엇입니까?")

## 3. 클래스 자체가 직렬화 가능한지 확인

`is_lc_serializable()`은 LangChain 객체가 직렬화를 지원하는지 알려주는 메서드다. 먼저 `ChatOpenAI` **클래스** 자체에 대해 호출해서 (인스턴스를 만들기 전에도) 직렬화가 가능한 종류의 클래스인지 확인한다.

In [20]:
# 클래스 자체에 대해 호출: ChatOpenAI가 직렬화를 지원하는 클래스인지 확인.
print(f"ChatOpenAI: {ChatOpenAI.is_lc_serializable()}")

ChatOpenAI: True


## 4. 인스턴스에 대해서도 확인

실제로 만든 `llm` 인스턴스에 대해서도 같은 메서드를 호출해본다. 결과는 마찬가지로 `True`.

In [21]:
llm = ChatOpenAI(model="gpt-5.6-luna", temperature=0)

# 인스턴스에 대해 호출해도 결과는 동일하게 True.
print(f"ChatOpenAI: {llm.is_lc_serializable()}")

ChatOpenAI: True


## 5. 체인 전체도 직렬화 가능한지 확인

`prompt | llm`으로 만든 체인 전체에 대해서도 `is_lc_serializable()`을 호출할 수 있다. 체인을 구성하는 모든 요소(prompt, llm)가 직렬화 가능해야 체인 전체도 `True`가 된다.

In [22]:
chain = prompt | llm

# 체인을 구성하는 요소들이 모두 직렬화 가능해야 체인 전체도 True가 된다.
chain.is_lc_serializable()

True

## 6. dumpd(): 체인을 dict로 직렬화

`dumpd(chain)`은 체인을 파이썬 `dict`로 변환해준다. 결과를 보면 `PromptTemplate`, `ChatOpenAI` 각각의 설정값(템플릿 문자열, 모델명 등)이 그대로 dict 안에 들어있다.

한 가지 중요한 점: `openai_api_key` 값이 실제 키 문자열이 아니라 `{"lc": 1, "type": "secret", "id": ["OPENAI_API_KEY"]}`처럼 **"이 값은 OPENAI_API_KEY라는 환경변수에서 가져온 비밀값이다"라는 참조 정보만** 들어있다. 덕분에 이 dict를 파일로 저장해도 API 키가 그대로 노출되지 않는다.

In [23]:
from langchain_core.load import dumpd, dumps

# 체인을 dict 형태로 직렬화. API 키는 실제 값이 아니라
# {"type": "secret", "id": ["OPENAI_API_KEY"]}처럼 "참조"만 저장되어 노출되지 않는다.
dumpd_chain = dumpd(chain)
dumpd_chain

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'runnable', 'RunnableSequence'],
 'kwargs': {'first': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'prompts', 'prompt', 'PromptTemplate'],
   'kwargs': {'input_variables': ['fruit'],
    'template': '{fruit}의 색상이 무엇입니까?',
    'template_format': 'f-string'},
   'name': 'PromptTemplate'},
  'last': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'chat_models', 'openai', 'ChatOpenAI'],
   'kwargs': {'model_name': 'gpt-5.6-luna',
    'openai_api_key': {'lc': 1, 'type': 'secret', 'id': ['OPENAI_API_KEY']},
    'stream_usage': True},
   'name': 'ChatOpenAI'}},
 'name': 'RunnableSequence'}

직렬화된 결과의 타입을 확인해보면 실제로 `dict`인 것을 알 수 있다.

In [24]:
type(dumpd_chain)  # dict

dict

## 7. dumps(): 체인을 JSON 문자열로 직렬화

`dumps(chain)`은 `dumpd`와 내용은 같지만, dict가 아니라 **JSON 형식의 문자열**로 바로 변환해준다. 파일에 텍스트로 저장하거나 네트워크로 전송할 때 편리하다.

In [25]:
dumps_chain = dumps(chain)
dumps_chain

'{"lc": 1, "type": "constructor", "id": ["langchain", "schema", "runnable", "RunnableSequence"], "kwargs": {"first": {"lc": 1, "type": "constructor", "id": ["langchain", "prompts", "prompt", "PromptTemplate"], "kwargs": {"input_variables": ["fruit"], "template": "{fruit}\\uc758 \\uc0c9\\uc0c1\\uc774 \\ubb34\\uc5c7\\uc785\\ub2c8\\uae4c?", "template_format": "f-string"}, "name": "PromptTemplate"}, "last": {"lc": 1, "type": "constructor", "id": ["langchain", "chat_models", "openai", "ChatOpenAI"], "kwargs": {"model_name": "gpt-5.6-luna", "openai_api_key": {"lc": 1, "type": "secret", "id": ["OPENAI_API_KEY"]}, "stream_usage": true}, "name": "ChatOpenAI"}}, "name": "RunnableSequence"}'

타입을 확인해보면 이번엔 `str`이다.

In [26]:
type(dumps_chain)  # str

str

## 8. pickle로 파일에 저장

파이썬 표준 라이브러리 `pickle`을 이용해서, 직렬화된 dict(`dumpd_chain`)를 `fruit_chain.pkl` 파일에 그대로 저장한다.

In [27]:
import pickle

# 직렬화된 dict를 pickle 파일로 저장.
with open("fruit_chain.pkl", "wb") as f:
    pickle.dump(dumpd_chain, f)

## 9. JSON 파일로도 저장

같은 dict를 이번엔 `json` 모듈로 `fruit_chain.json` 파일에 저장한다. JSON은 pickle과 달리 사람이 읽을 수 있는 텍스트 형식이고, 파이썬이 아닌 다른 언어/도구에서도 쉽게 읽을 수 있다는 장점이 있다.

In [28]:
import json

# 같은 dict를 이번엔 JSON 파일로 저장 (사람이 읽을 수 있는 텍스트 형식).
with open("fruit_chain.json", "w") as fp:
    json.dump(dumpd_chain, fp)

## 10. pickle 파일 다시 불러오기

저장해둔 `fruit_chain.pkl`을 다시 읽어서, 직렬화 당시의 dict(`loaded_chain`)를 그대로 복원한다. 아직은 실행 가능한 체인이 아니라 평범한 dict 상태다.

In [29]:
with open("fruit_chain.pkl", "rb") as f:
    # pickle 파일을 읽어서 저장 당시의 dict를 그대로 복원.
    loaded_chain = pickle.load(f)

## 11. dict를 다시 실행 가능한 체인으로 역직렬화

`load()`는 `dumpd()`의 반대 동작을 한다: dict를 다시 실제 LangChain 객체(여기서는 `RunnableSequence` 체인)로 복원해준다. `allowed_objects="all"`은 "신뢰할 수 있는 데이터이니 어떤 타입의 객체든 복원해도 좋다"는 허용 옵션이다 (신뢰할 수 없는 데이터를 역직렬화하면 임의 코드 실행 위험이 있을 수 있어서, LangChain이 기본적으로는 이를 제한한다).

여기서는 `secrets_map`을 따로 넘기지 않았는데도 정상적으로 `OPENAI_API_KEY`를 찾아서 사용한다. `load_dotenv()`로 이미 환경변수에 올려둔 값을 LangChain이 알아서 참조하기 때문이다. 복원된 `chain_from_file`을 실제로 호출해보면 원래 체인과 동일하게 동작한다.

In [30]:
from langchain_core.load import load

# dict -> 실행 가능한 체인 객체로 복원.
# allowed_objects="all" : 신뢰할 수 있는 데이터이므로 모든 타입의 객체 복원을 허용.
chain_from_file = load(loaded_chain, allowed_objects="all")

print(chain_from_file.invoke({"fruit": "사과"}))

content='사과의 색상은 보통 빨간색이지만, 품종에 따라 초록색이나 노란색 등 다양합니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 15, 'total_tokens': 47, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EOwymqMSe2yN2BY1rfQ6K7Fz0c9Tj', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0ad62-4a3d-7341-a6f7-69fef4bd0e92-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 32, 'total_tokens': 47, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## 12. secrets_map을 명시적으로 전달하기

앞선 예제는 환경변수에서 자동으로 키를 찾아줬지만, `secrets_map` 파라미터로 "어떤 비밀값 id를 실제로 어떤 값에 매핑할지"를 명시적으로 전달할 수도 있다. 이렇게 하면 환경변수 이름이 다르거나, 환경변수에 의존하지 않는 다른 실행 환경으로 옮길 때도 명확하게 키를 지정해줄 수 있다.

In [ ]:
from langchain_core.load import load

# secrets_map으로 "OPENAI_API_KEY"라는 참조를 실제 키 값에 명시적으로 연결해준다.
load_chain = load(
    loaded_chain, secrets_map={"OPENAI_API_KEY": os.environ["OPENAI_API_KEY"]}, allowed_objects="all"
)

print(load_chain.invoke({"fruit": "사과"}))

content='사과의 색상은 보통 빨간색이지만, 품종에 따라 초록색이나 노란색도 있습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 15, 'total_tokens': 46, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EOx3yT7uohnDLZTVeQKrtGfA9WENy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0ad67-360e-7f83-83bc-07cb77acaa3d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 31, 'total_tokens': 46, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## 13. JSON 파일에서부터 전체 과정 다시 확인

이번엔 pickle이 아니라 JSON 파일(`fruit_chain.json`)에서 시작해서 같은 과정을 반복한다: JSON 파일을 읽어 dict로 만들고(`json.load`), 그 dict를 다시 체인 객체로 복원한다(`load`).

In [39]:
with open("fruit_chain.json", "r") as fp:
    # JSON 파일 -> dict로 읽기
    loaded_from_json_chain = json.load(fp)
    # dict -> 실행 가능한 체인 객체로 복원
    loads_chain = load(loaded_from_json_chain, allowed_objects="all")

JSON 파일로부터 복원한 체인도 pickle로 복원했을 때와 동일하게 정상적으로 호출된다. 즉 pickle이든 JSON이든, `dumpd()`로 만든 dict를 저장해뒀다가 `load()`로 복원하면 똑같이 동작하는 체인을 다시 만들어낼 수 있다.

In [40]:
print(loads_chain.invoke({"fruit": "사과"}))

content='사과의 색상은 보통 빨간색, 초록색, 노란색 등이 있습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 15, 'total_tokens': 40, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EOx4LnAzAZYv1eiRhUy0DQXmY4HSi', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0ad67-93a8-7f62-a480-9acf5432ae45-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 25, 'total_tokens': 40, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
